# H2O Higgs Boson Classification

**Tabular Binary Classification** — Distinguish Higgs boson signal from background noise.

Models: CatBoost · LightGBM · XGBoost  
Baselines: LazyPredict · FLAML AutoML  
Dataset: Higgs Boson (250K rows × 33 columns, sampled to 50K)  
Target: `Label` (s=signal, b=background)

## 2 · Project Overview

We classify particle physics events as **signal (s)** — produced by a Higgs boson decay — or **background (b)** — standard model processes.

This is a famous Kaggle competition dataset from CERN, featuring 30 physics-derived features.

## 3 · Learning Objectives

1. Handle a large-scale physics dataset.
2. Deal with missing values encoded as -999.0.
3. Build binary classifiers for scientific discovery.
4. Understand feature importance in particle physics.
5. Compare boosting models on 50K samples.

## 4 · Problem Statement

Given 30 kinematic features from particle collisions at the LHC, classify each event as a **Higgs boson signal (s)** or **background noise (b)**.

## 5 · Why This Project Matters

- The Higgs boson discovery won the 2013 Nobel Prize in Physics.
- ML plays a critical role in particle physics analysis.
- Feature engineering in physics is domain-driven.
- This teaches handling of missing-value sentinels (-999.0).

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 250,000 (sampled to 50,000) |
| **Columns** | 33 |
| **Derived features** | DER_* (13 physics-derived) |
| **Primitive features** | PRI_* (17 raw measurements) |
| **Target** | Label (s=signal, b=background) |
| **Missing sentinel** | -999.0 |
| **Source** | Local training.csv / Kaggle |

## 7 · Dataset Source and License Notes

- **Source**: Kaggle Higgs Boson Machine Learning Challenge (2014).
- **Organizer**: ATLAS experiment at CERN.
- **Reference**: Adam-Bourdarios et al. (2014).
- **License**: Open for research and education.

## 8 · Environment Setup

In [ ]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

## 9 · Imports

In [ ]:
import os, json, time, warnings, random, re, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

## 10 · Configuration / Constants

In [ ]:
TARGET = "Label"
MAX_ROWS = 50000
DATA_PATH = os.path.join(os.path.dirname(os.path.abspath("__file__")), "training.csv")
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Data path: {DATA_PATH}")
print(f"Save dir : {SAVE_DIR}")

## 11 · Dataset Download or Loading

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Full dataset: {df.shape}")

if len(df) > MAX_ROWS:
    df = df.sample(n=MAX_ROWS, random_state=SEED).reset_index(drop=True)
    print(f"Sampled to: {df.shape}")

df.head()

## 12 · Data Validation Checks

In [ ]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nMissing values (NaN): {df.isnull().sum().sum()}")
# Count -999.0 sentinel values
sentinel_count = (df.select_dtypes(include="number") == -999.0).sum()
print(f"\nSentinel (-999.0) values per column:")
for col, cnt in sentinel_count[sentinel_count > 0].items():
    print(f"  {col}: {cnt}")
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}")
assert TARGET in df.columns
print(f"Shape: {df.shape}")

## 13 · Exploratory Data Analysis

In [ ]:
# Feature distributions (skip EventId and Weight)
feature_cols = [c for c in df.columns if c not in [TARGET, "EventId", "Weight"]]
der_cols = [c for c in feature_cols if c.startswith("DER")]
pri_cols = [c for c in feature_cols if c.startswith("PRI")]

fig, axes = plt.subplots(3, 5, figsize=(20, 12))
for i, col in enumerate(der_cols[:15]):
    ax = axes[i // 5, i % 5]
    valid = df[df[col] != -999.0][col]
    valid.hist(bins=30, ax=ax, color="steelblue", edgecolor="black", alpha=0.8)
    ax.set_title(col, fontsize=8)
plt.suptitle("Derived Feature Distributions (excluding -999)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation of features with target
df_corr = df.copy()
df_corr[TARGET] = (df_corr[TARGET] == "s").astype(int)
# Replace -999 with NaN for correlation
num_cols = df_corr.select_dtypes(include="number").columns
df_corr[num_cols] = df_corr[num_cols].replace(-999.0, np.nan)
corr_target = df_corr[num_cols].corr()[TARGET].drop([TARGET, "EventId", "Weight"], errors="ignore").abs().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(12, 6))
corr_target.head(15).plot(kind="bar", ax=ax, color="steelblue", edgecolor="black")
ax.set_title("Top 15 Features Correlated with Signal")
ax.set_ylabel("|Correlation|")
plt.tight_layout()
plt.show()

## 14 · Target Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
df[TARGET].value_counts().plot(kind="bar", ax=ax, color=["steelblue", "coral"], edgecolor="black")
ax.set_title(f"Target Distribution: {TARGET}")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

for val in df[TARGET].value_counts().index:
    n = (df[TARGET] == val).sum()
    print(f"  {val}: {n} ({100*n/len(df):.1f}%)")

## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 split after encoding target.

In [ ]:
# Prepare features
X = df.drop(columns=[TARGET, "EventId", "Weight"])
y_raw = df[TARGET].values

# Encode target: s=1, b=0
le = LabelEncoder()
y = le.fit_transform(y_raw)
print(f"Classes: {list(le.classes_)} → [0, 1]")

# Replace -999.0 with NaN, then fill with median
X = X.replace(-999.0, np.nan)
for col in X.columns:
    X[col] = X[col].fillna(X[col].median())

# Sanitize column names
X.columns = [re.sub(r"[^A-Za-z0-9_]", "_", c) for c in X.columns]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 16 · Preprocessing Strategy

- **Missing sentinels**: -999.0 replaced with NaN, then imputed with column median.
- **Dropped**: EventId (identifier), Weight (competition scoring only).
- **Scaling**: Not needed for tree models.
- **Target**: s→1, b→0 via LabelEncoder.

## 17 · Feature Engineering

Physics features are already domain-engineered (DER_* derived variables). No additional feature engineering needed — domain scientists designed these features.

## 18 · Baseline Model

In [ ]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

acc_bl = accuracy_score(y_test, y_pred_bl)
f1_bl = f1_score(y_test, y_pred_bl, average="binary")

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {acc_bl:.4f}")
print(f"F1       : {f1_bl:.4f}")
print(f"\n{classification_report(y_test, y_pred_bl, target_names=le.classes_)}")

## 19 · LazyPredict Benchmark

Subsampled to 10K for LazyPredict speed.

In [ ]:
from lazypredict.Supervised import LazyClassifier

# Subsample for LazyPredict speed
X_lp, _, y_lp, _ = train_test_split(X_train, y_train, train_size=8000, random_state=SEED, stratify=y_train)
X_lp_test, _, y_lp_test, _ = train_test_split(X_test, y_test, train_size=2000, random_state=SEED, stratify=y_test)

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_lp, X_lp_test, y_lp, y_lp_test)

print("LazyPredict — Top 15 Classifiers (subsample):")
print(lazy_models.head(15).to_string())

## 20 · FLAML AutoML Run

In [ ]:
from flaml import AutoML

try:
    flaml_automl = AutoML()
    flaml_automl.fit(X_train, y_train, task="classification", time_budget=60,
                     metric="f1", verbose=0, seed=SEED)
    y_pred_flaml = flaml_automl.predict(X_test)
    acc_flaml = accuracy_score(y_test, y_pred_flaml)
    f1_flaml = f1_score(y_test, y_pred_flaml)
    print(f"FLAML best: {flaml_automl.best_estimator}")
    print(f"Accuracy: {acc_flaml:.4f}, F1: {f1_flaml:.4f}")
except Exception as e:
    print(f"FLAML failed: {e}")
    print("Using baseline predictions as FLAML fallback.")
    y_pred_flaml = y_pred_bl

## 21 · Additional Models: CatBoost, LightGBM, XGBoost

In [ ]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostClassifier
    t0 = time.perf_counter()
    cb = CatBoostClassifier(iterations=200, learning_rate=0.05, depth=6,
                            task_type="GPU", devices="0", verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test)
    print(f"CatBoost F1: {f1_score(y_test, results['CatBoost'], average='binary'):.4f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    lg = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, max_depth=6,
                            device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
    lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
           callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM F1: {f1_score(y_test, results['LightGBM'], average='binary'):.4f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBClassifier
    t0 = time.perf_counter()
    xgb_model = XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=6,
                              device="cuda", tree_method="hist", verbosity=0,
                              eval_metric="mlogloss", n_jobs=-1, random_state=SEED)
    xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost F1: {f1_score(y_test, results['XGBoost'], average='binary'):.4f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost: {e}")
gpu_cleanup()

## 22 · Top 3 Model Selection

In [ ]:
# Add baseline and FLAML
results["Baseline"] = y_pred_bl
results["FLAML"] = y_pred_flaml

model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average='binary'), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3: {top3_names}")

## 23 · Final Training and Evaluation of Top 3

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], colorbar=False)
    f1 = f1_score(y_test, yp, average='binary')
    axes[i].set_title(f"{name}\nF1={f1:.4f}")

plt.suptitle("Top 3 — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    y_t = y_test if isinstance(y_test, np.ndarray) else y_test.values
    print(f"\n  {name}:")
    print(f"    Accuracy: {accuracy_score(y_t, yp):.4f}")
    print(f"    F1      : {f1_score(y_t, yp, average='binary'):.4f}")
    print(f"    Precision: {precision_score(y_t, yp, average='binary'):.4f}")
    print(f"    Recall  : {recall_score(y_t, yp, average='binary'):.4f}")

## 24 · Error Analysis

In [ ]:
best_name = top3_names[0]
best_pred = results[best_name]

y_t = y_test if isinstance(y_test, np.ndarray) else y_test.values
misclassed = (y_t != best_pred)
n_wrong = misclassed.sum()
print(f"Best model: {best_name}")
print(f"Misclassified: {n_wrong} / {len(y_t)} ({100*n_wrong/len(y_t):.1f}%)")

if n_wrong > 0:
    print(f"\nClassification Report ({best_name}):")
    print(classification_report(y_t, best_pred))

## 25 · Interpretation and Business Insight

**Key findings:**
- **DER_mass_MMC** (reconstructed mass) is the strongest feature — as expected physically.
- **DER_mass_transverse_met_lep** and **DER_mass_vis** are also strong discriminators.
- **Primitive features** (PRI_*) are less predictive individually.
- Missing values (-999.0) concentrate in jet-related features when fewer jets are detected.

**Physics takeaway:** The Higgs boson signature is most visible in reconstructed mass distributions, consistent with H→ττ decay mode.

## 26 · Limitations

1. Sampled to 50K from 250K — full dataset may yield better results.
2. Simplified to binary — real analysis has nuanced background categories.
3. No systematic uncertainty handling.
4. Weight column ignored (affects competition metric AMS).
5. Features are pre-computed — no access to raw detector data.

## 27 · How to Improve This Project

1. Train on full 250K rows.
2. Use the competition AMS metric with Weight.
3. Use physics-informed feature engineering.
4. Try deep neural networks (as done by winning solution).
5. Handle -999.0 with indicator features instead of median imputation.

## 28 · Production Considerations

- Real physics analyses use ensemble of models.
- Systematic uncertainties must be propagated.
- Production runs on the CERN computing grid.
- Results require formal statistical validation (CLs method).

## 29 · Common Mistakes

1. Not handling -999.0 sentinel values.
2. Including EventId or Weight as features.
3. Using accuracy on slightly imbalanced data.
4. Not stratifying the train/test split.
5. Ignoring physics context when interpreting features.

## 30 · Mini Challenge / Exercises

1. Add indicator features for -999.0 values — does it help?
2. Use only DER_* features — how much accuracy is lost?
3. Train on 100K rows — significant improvement?
4. Plot ROC curves for all models.
5. Implement the AMS metric from the competition.

## 31 · Final Summary / Key Takeaways

1. **Reconstructed mass** features are most discriminative for Higgs detection.
2. **Boosting models** are the dominant approach in particle physics ML.
3. **-999.0 handling** is critical for this dataset.
4. **Domain knowledge** from physics designed the best features.
5. **ML enabled** the Higgs boson discovery verification at CERN.

## Save Metrics

In [ ]:
metrics_out = {}
for name, yp in results.items():
    y_t = y_test if isinstance(y_test, np.ndarray) else y_test.values
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_t, yp)), 4),
        "f1": round(float(f1_score(y_t, yp, average='binary')), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))